# M4 — multi-target ATT&CK (multi-label) sobre threat

**Pregunta:** ¿cómo nos va prediciendo *técnicas* ATT&CK en vez de tiers?

**Hallazgo EDA previo (verificado en este notebook):** en `signals` live solo hay 4 técnicas: T1190 (114,507, todo threat), T1041 (7,742), T1567 (7,728 = todos los malicious), T1204 (20). suspicious = `[T1190]` (1 label), malicious = `[T1190,T1041,T1567]` (3 labels, 20 filas con +T1204). En `incident_signals` histórico aparecen 2 más: T1486 (161) y T1566 (3), casi todo `Data Theft`.

O sea: la cabeza es casi determinista del tier (T1567 ⟺ malicious) y hay **cola larga** (T1204/T1486/T1566). El experimento mide si un multi-label lineal captura la cabeza y qué pasa en la cola.

Diseño: solo filas **threat** (suspicious+malicious, 114,521 — benign no tiene técnicas y M1 ya lo filtra), targets = 6 técnicas [T1190,T1041,T1567,T1204,T1486,T1566], split **temporal 80/20** igual que M1/M2/M3, modelo `OneVsRest(LogisticRegression)` sobre TF-IDF 5k + tabular P1–P3, baseline oráculo-tier (predice técnicas desde el tier real).

In [1]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent
print('ROOT:', ROOT)

COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques',
        'message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
df = pd.read_parquet(ROOT / 'data' / 'signals' / 'signals.parquet', columns=COLS)
th = df[df.label_binary != 'benign'].copy().reset_index(drop=True)
print('threat:', th.shape, th.label_binary.value_counts().to_dict())

TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except Exception:
        return []
th['tech_list'] = th.attack_techniques.fillna('').map(parse)
c = Counter(t for ts in th.tech_list for t in ts)
print('tecnicas en live-threat:', {k: c.get(k,0) for k in TECHS})
print('n-labels por evento:', th.tech_list.map(len).value_counts().to_dict())

ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


threat: (114521, 14) {'suspicious': 106793, 'malicious': 7728}


tecnicas en live-threat: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}
n-labels por evento: {1: 106793, 3: 7708, 4: 20}


In [2]:
# features P1-P3 (igual que M1) + split temporal 80/20
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
th['ts'] = pd.to_datetime(th['timestamp'], unit='s', utc=True)
th['hour'] = th.ts.dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp'] - th['event_time']) / 3600).clip(-1e4, 1e4).fillna(-1)
for col in CATS:
    th[col] = th[col].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.8)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()}  corte: {pd.to_datetime(cutoff, unit="s", utc=True)}')

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=TECHS)
Y = mlb.fit_transform(th.tech_list)
print('Y:', Y.shape, 'densidad:', round(Y.mean(), 4))

train/test: 91617/22904  corte: 2024-07-27 05:13:54.964786053+00:00


Y: (114521, 6) densidad: 0.1892


In [3]:
# X: TF-IDF 5k (fit solo train) + tabular one-hot
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS),
                           ('num', 'passthrough', ['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']])
Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
Xtr = hstack([Ttr, Atr]).tocsr(); Xte = hstack([Tte, Ate]).tocsr()
print('X:', Xtr.shape, Xte.shape)

X: (91617, 5054) (22904, 5054)


In [4]:
# modelo: OneVsRest LogReg balanceado + baseline oraculo-tier
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

t0 = time.time()
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced'))
clf.fit(Xtr, Y[tr])
P = clf.predict(Xte)
print(f'fit+predict: {time.time()-t0:.0f}s')

# oraculo-tier: suspicious->[T1190], malicious->[T1190,T1041,T1567] (+T1204 impredecible)
oracle = np.zeros_like(Y[te])
lab_te = th[te].label_binary.to_numpy()
oracle[:, TECHS.index('T1190')] = 1
mal = lab_te == 'malicious'
oracle[mal, TECHS.index('T1041')] = 1
oracle[mal, TECHS.index('T1567')] = 1
print('oraculo listo')

C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


fit+predict: 56s
oraculo listo


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 5 is present in all training examples.
  warnings.warn(


In [5]:
from sklearn.metrics import (classification_report, f1_score, hamming_loss,
                             accuracy_score, average_precision_score)
print('=== M4 OneVsRest-LogReg ===')
print(classification_report(Y[te], P, target_names=TECHS, zero_division=0))
print('micro-F1:', round(f1_score(Y[te], P, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], P, average='macro'), 4))
print('hamming:', round(hamming_loss(Y[te], P), 4),
      '| subset-acc:', round(accuracy_score(Y[te], P), 4))
print('=== baseline oraculo-tier ===')
print('micro-F1:', round(f1_score(Y[te], oracle, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], oracle, average='macro'), 4),
      '| hamming:', round(hamming_loss(Y[te], oracle), 4),
      '| subset-acc:', round(accuracy_score(Y[te], oracle), 4))
for j, t in enumerate(TECHS):
    try:
        ap = average_precision_score(Y[te][:, j], P[:, j])
    except Exception:
        ap = 0.0
    print(f'{t}: soporte test {int(Y[te][:, j].sum())}  PR-AUC {round(float(ap), 4)}')

=== M4 OneVsRest-LogReg ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.12      0.10      3733
       T1567       0.08      0.11      0.10      3721
       T1204       0.19      1.00      0.32        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.72      0.78      0.75     30356
   macro avg       0.23      0.37      0.25     30356
weighted avg       0.77      0.78      0.78     30356
 samples avg       0.86      0.90      0.82     30356

micro-F1: 0.7484 | macro-F1: 0.2516
hamming: 0.1162 | subset-acc: 0.6451
=== baseline oraculo-tier ===
micro-F1: 0.9994 | macro-F1: 0.4997 | hamming: 0.0002 | subset-acc: 0.999
T1190: soporte test 22892  PR-AUC 1.0
T1041: soporte test 3733  PR-AUC 0.1537
T1567: soporte test 3721  PR-AUC 0.1537
T1204: soporte test 10  PR-AUC 0.1887
T1486: soporte test 0  PR-AUC 0.0
T1566: s

C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-p

## Lectura: cómo nos va con multi-target (medido)

- **T1190: P/R/F1 1.00/1.00/1.00 (PR-AUC 1.0).** Trivial: todo threat lo lleva. No mide nada, confirma el EDA.
- **T1041/T1567: P 0.08 / R 0.11-0.12 / F1 0.10, PR-AUC 0.15.** El lineal convergido ni dispara: no separa malicious de suspicious con estos features. Contraste clave con M2-XGBoost (P 0.79/R 0.97 en malicious): la no-linealidad + texto sí separa. (Ojo: sin converger daba R~0.95/P~0.22 — inestable en ambas direcciones, otra señal de que el lineal no sirve aquí.)
- **T1204: R 1.00 (10/10) con P 0.19.** Atrapa los 10 a ciegas, a base de falsos positivos.
- **T1486/T1566: F1 0, sin soporte en live.** No hay de dónde aprender; el oráculo-tier también da 0 ahí (su macro es 0.4997 por eso).
- **micro-F1 0.75 vs macro-F1 0.25, hamming 0.12, subset-acc 0.65 (oráculo: 0.9994/0.4997/0.0002/0.999).** El micro lo domina T1190; el macro castiga cabezas fallidas + cola. Reportar ambos o el paper miente.
- **Siguiente paso:** M4 como segunda cabeza sobre M2 (mismo X, otro Y) para la demo; la cola (T1204/T1486/T1566) pide few-shot/LLM con `incident_signals` histórico o reglas. No prometer ATT&CK por evento en la demo: vender cabeza T1190, documentar resto.

In [6]:
import json
m = {'model': 'm4-multilabel-v1', 'labels': TECHS,
     'micro_f1': round(float(f1_score(Y[te], P, average='micro')), 4),
     'macro_f1': round(float(f1_score(Y[te], P, average='macro')), 4),
     'hamming': round(float(hamming_loss(Y[te], P)), 4),
     'subset_acc': round(float(accuracy_score(Y[te], P)), 4),
     'n_train': int(tr.sum()), 'n_test': int(te.sum()),
     'oracle': {'micro_f1': round(float(f1_score(Y[te], oracle, average='micro')), 4),
                'macro_f1': round(float(f1_score(Y[te], oracle, average='macro')), 4)}}
(ROOT / 'models' / 'm4_metrics.json').write_text(json.dumps(m, indent=2))
print('guardado models/m4_metrics.json', m)

guardado models/m4_metrics.json {'model': 'm4-multilabel-v1', 'labels': ['T1190', 'T1041', 'T1567', 'T1204', 'T1486', 'T1566'], 'micro_f1': 0.7484, 'macro_f1': 0.2516, 'hamming': 0.1162, 'subset_acc': 0.6451, 'n_train': 91617, 'n_test': 22904, 'oracle': {'micro_f1': 0.9994, 'macro_f1': 0.4997}}


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
